# 📤 Output Parsers and Structured Output in LangChain

## Learning Objectives
In this notebook, you will learn:
1. **String Output Parsing** - Extract plain text output from an LLM using `StrOutputParser`
2. **JSON Output Parsing** - Parse loosely-structured JSON responses with `JsonOutputParser`
3. **Pydantic Output Parsing** - Enforce type-safe schemas with `PydanticOutputParser` and format instructions
4. **Modern Structured Output** - Use `with_structured_output()` to bind a Pydantic schema directly to the model
5. **Complex Nested Schemas** - Extract multi-level structured data (objects containing objects and lists)

## Prerequisites
- `OPENAI_API_KEY` set in a `.env` file at the project root
- Basic familiarity with LangChain Expression Language (LCEL) `prompt | model | parser` chains
- Basic familiarity with Pydantic `BaseModel` and `Field`

---
## 🔧 Part 0: Environment Setup

We import the output-parser classes we'll compare, load API keys from `.env`, and create a single shared `ChatOpenAI` model instance that every demo function below reuses. Keeping one model instance is intentional — it isolates the differences between parsing strategies, not the model configuration.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Imports and Model Initialization
# ============================================================================
from typing import List, Optional

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from langchain_core.output_parsers import (
    StrOutputParser,
    JsonOutputParser,
    PydanticOutputParser,
)
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Load OPENAI_API_KEY (and any other keys) from .env
load_dotenv()

# Single shared model instance reused by every demo in this notebook
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"🤖 LLM initialized: {model.model_name}")

---
## 📝 Part 1: Basic Output Parsers

LangChain ships three classic output parser types that sit at the end of an LCEL chain and coerce the raw LLM response into something usable — a string, a dict, or a validated Pydantic object. Each demo below uses the exact same `prompt | model | parser` shape so you can compare what each parser actually changes.

### Key Concepts:
- **StrOutputParser**: Extracts just the text content from the model's response.
- **JsonOutputParser**: Parses a JSON-formatted response into a Python `dict`.
- **PydanticOutputParser**: Validates and parses the response into a typed Pydantic model, using format instructions injected into the prompt.

### 1.1 🔤 String Output Parser

The simplest parser in LangChain — it takes the model's raw `AIMessage` and returns just the text content as a plain Python `str`. Useful whenever you don't need structure, just the answer.

In [ ]:
# ============================================================================
# STRING OUTPUT PARSER: Basic StrOutputParser Demo
# ============================================================================
def demo_str_parser():
    """Basic string output parser."""

    prompt = ChatPromptTemplate.from_template(
        "Give me a one-word answer: What color is the sky?"
    )
    parser = StrOutputParser()

    chain = prompt | model | parser

    result = chain.invoke({})
    print(f"Result: '{result}' (type: {type(result).__name__})")

### 1.2 🧩 JSON Output Parser

`JsonOutputParser` expects the model to return a JSON-formatted string and parses it into a Python `dict`. This works well for simple, flat structures, but there is no schema validation — the model can omit keys or return the wrong types.

In [ ]:
# ============================================================================
# JSON OUTPUT PARSER: Parsing JSON Responses
# ============================================================================
def demo_json_parser():
    """JSON output parser."""

    prompt = ChatPromptTemplate.from_template(
        "Return a JSON object with keys 'city' and 'country' for: {place}\n"
        "Return ONLY valid JSON, no explanation."
    )
    parser = JsonOutputParser()

    chain = prompt | model | parser

    result = chain.invoke({"place": "The Eiffel Tower"})
    print(f"Result: {result}")
    print(f"City: {result['city']}, Country: {result['country']}")

### 1.3 🛡️ Pydantic Output Parser

`PydanticOutputParser` adds real type safety: define a schema with `BaseModel` and `Field`, inject its `get_format_instructions()` into the prompt, and get back a validated, typed object instead of a raw dict.

In [ ]:
# ============================================================================
# PYDANTIC OUTPUT PARSER: Type-Safe Structured Data
# ============================================================================
def demo_pydantic_parser():
    """Pydantic output parser for type-safe structured data."""

    # Define schema
    class Recipe(BaseModel):
        name: str = Field(description="Name of the recipe")
        ingredients: List[str] = Field(description="List of ingredients")
        prep_time_minutes: int = Field(description="Preparation time in minutes")
        difficulty: str = Field(description="easy, medium, or hard")

    parser = PydanticOutputParser(pydantic_object=Recipe)

    prompt = ChatPromptTemplate.from_template(
        "Create a simple recipe for: {dish}\n\n{format_instructions}"
    ).partial(format_instructions=parser.get_format_instructions())

    chain = prompt | model | parser

    result = chain.invoke({"dish": "scrambled eggs"})
    print(f"Recipe: {result.name}")
    print(f"Ingredients: {result.ingredients}")
    print(f"Prep time: {result.prep_time_minutes} mins")
    print(f"Difficulty: {result.difficulty}")

    # Type-safe access
    print(
        f"\nType check - prep_time is int: {isinstance(result.prep_time_minutes, int)}"
    )

---
## 🏗️ Part 2: Modern Structured Output

`PydanticOutputParser` works, but it requires manually injecting format instructions into the prompt and hoping the model follows them. Most current chat models support native structured output instead — `with_structured_output()` binds a schema directly to the model so it always returns a validated object, with no manual format instructions needed.

> **Key Insight**: Prefer `with_structured_output()` over `PydanticOutputParser` for any model that supports it (OpenAI, Anthropic, and most modern providers) — it is more reliable because the schema is enforced by the provider's API, not just requested in the prompt text.

### 2.1 🎯 Structured Output Extraction

`model.with_structured_output(Schema)` returns a new runnable that always outputs an instance of `Schema` (or raises) — no separate parser step in the chain at all.

In [ ]:
# ============================================================================
# STRUCTURED OUTPUT: with_structured_output() Method
# ============================================================================
def demo_structured_output():
    """Modern with_structured_output() method."""

    class TaskExtraction(BaseModel):
        """Extracted task information."""

        task: str = Field(description="The main task to do")
        priority: str = Field(description="high, medium, or low")
        deadline: Optional[str] = Field(description="Deadline if mentioned")
        assignee: Optional[str] = Field(description="Person assigned if mentioned")

    # Bind schema to model
    structured_model = model.with_structured_output(TaskExtraction)

    # No need for format instructions - it's automatic
    prompt = ChatPromptTemplate.from_template("Extract task information from: {text}")

    chain = prompt | structured_model

    texts = [
        "John needs to finish the report by Friday - it's urgent",
        "We should update the docs sometime next week",
        "Critical: Fix the login bug ASAP",
    ]

    print("Task Extractions:")
    for text in texts:
        result = chain.invoke({"text": text})
        print(f"\nInput: {text}")
        print(f"  Task: {result.task}")
        print(f"  Priority: {result.priority}")
        print(f"  Deadline: {result.deadline}")
        print(f"  Assignee: {result.assignee}")

### 2.2 🏢 Complex Nested Schemas

Structured output isn't limited to flat schemas — a Pydantic model can nest other Pydantic models and lists, and `with_structured_output()` will still parse and validate the entire nested structure.

In [ ]:
# ============================================================================
# COMPLEX SCHEMA: Nested Pydantic Models
# ============================================================================
def demo_complex_schema():
    """Complex nested schema with structured output."""

    class Address(BaseModel):
        street: str
        city: str
        country: str

    class Company(BaseModel):
        name: str
        industry: str
        employee_count: int
        headquarters: Address
        products: List[str]

    structured_model = model.with_structured_output(Company)

    prompt = ChatPromptTemplate.from_template(
        "Extract company information from: {text}"
    )

    chain = prompt | structured_model

    result = chain.invoke(
        {
            "text": "Apple Inc. is a tech company with 160,000 employees based in "
            "Cupertino, California, USA. They make iPhones, MacBooks, and iPads."
        }
    )

    print(f"Company: {result.name}")
    print(f"Industry: {result.industry}")
    print(f"Employees: {result.employee_count}")
    print(f"HQ: {result.headquarters.city}, {result.headquarters.country}")
    print(f"Products: {result.products}")

---
## 🎯 Part 3: Practice Exercise

Apply everything from Part 2 to a new domain: extracting structured movie information from a free-text review, including a numeric field with validation constraints (`ge`/`le`).

### 3.1 🎬 Movie Information Extraction

**Exercise**: build a schema and chain that extracts, from a movie review:
- Movie title
- Year released
- Director
- Main actors (list)
- Genre
- Rating (1-10, validated)

In [ ]:
# ============================================================================
# EXERCISE: Structured Movie Data Extraction
# ============================================================================
def exercise_structured_extraction():
    """
    EXERCISE: Create a schema and chain that extracts:
    - Movie title
    - Year released
    - Director
    - Main actors (list)
    - Genre
    - Rating (1-10)

    Test with a movie description.
    """

    class Movie(BaseModel):
        title: str = Field(description="Movie title")
        year: int = Field(description="Year released")
        director: str = Field(description="Director name")
        actors: List[str] = Field(description="Main actors")
        genre: str = Field(description="Primary genre")
        rating: int = Field(description="Rating from 1-10", ge=1, le=10)

    structured_model = model.with_structured_output(Movie)

    prompt = ChatPromptTemplate.from_template(
        "Extract movie information from this review:\n\n{review}"
    )

    chain = prompt | structured_model

    result = chain.invoke(
        {
            "review": "The Dark Knight (2008) directed by Christopher Nolan is an "
            "absolute masterpiece. Christian Bale and Heath Ledger deliver "
            "incredible performances in this action thriller. 10/10!"
        }
    )

    print(f"Title: {result.title}")
    print(f"Year: {result.year}")
    print(f"Director: {result.director}")
    print(f"Actors: {result.actors}")
    print(f"Genre: {result.genre}")
    print(f"Rating: {result.rating}/10")

---
## ▶️ Part 4: Running All Demos

The cell below mirrors the original script's `if __name__ == "__main__":` guard — Jupyter already sets `__name__` to `"__main__"`, so it runs unchanged. It calls every demo function defined above in sequence, making a real OpenAI API call each time.

> **Note**: Running this cell makes 6 separate calls to the OpenAI API and will incur a small cost.

In [ ]:
# ============================================================================
# RUN: Execute All Demos Sequentially
# ============================================================================
if __name__ == "__main__":
    print("=" * 50)
    print("Demo 1: String Parser")
    print("=" * 50)
    demo_str_parser()

    print("\n" + "=" * 50)
    print("Demo 2: JSON Parser")
    print("=" * 50)
    demo_json_parser()

    print("\n" + "=" * 50)
    print("Demo 3: Pydantic Parser")
    print("=" * 50)
    demo_pydantic_parser()

    print("\n" + "=" * 50)
    print("Demo 4: Structured Output (Modern)")
    print("=" * 50)
    demo_structured_output()

    print("\n" + "=" * 50)
    print("Demo 5: Complex Schema")
    print("=" * 50)
    demo_complex_schema()

    print("\n" + "=" * 50)
    print("Exercise: Movie Extraction")
    print("=" * 50)
    exercise_structured_extraction()

---
## 📝 Summary

In this notebook, we compared five ways to get structured or typed output from an LLM.

### 1. Classic Output Parsers
- **StrOutputParser**: plain text, no structure.
- **JsonOutputParser**: parses JSON into a `dict`, no validation.
- **PydanticOutputParser**: validated, typed output, but requires manually injecting format instructions into the prompt.

### 2. Modern Structured Output
- **with_structured_output()**: binds a Pydantic schema directly to the model — no manual format instructions, no separate parser step, and it handles nested schemas (objects and lists) automatically.
- Prefer this over `PydanticOutputParser` whenever the model supports it.

### Next Steps
- Continue to `07_chains_v1.ipynb` to see these parsers composed into larger multi-step LCEL chains.
- Note: this notebook overlaps significantly with `05_output_parsers_demo.ipynb` in the same folder — worth reviewing both together for possible consolidation.